# ChatGPT lab — from a base model to an assistant (ELELEM Chile 2026, Day 1)

**Time:** 1 h 40 min · **Runs in:** Google Colab — nothing to install · **Needs:** `OPENAI_API_KEY` in Colab secrets (🔑, from the pre-work)

This morning we followed one model family from n-grams to ChatGPT. This lab lets you *see* each step:

| Part | What you see | Practice | ≈ min |
|---|---|---|---|
| 1 | A **base model** (GPT-2, running in this notebook) only continues text — and you can read its next-token probabilities | 1 | 20 |
| 2 | An **instruction-tuned** model answers questions; the system prompt shapes it | 2 | 10 |
| 3 | The assistant is still a next-token predictor: **log-probabilities and temperature** | 3 | 15 |
| 4 | **Tokens**, context and cost | 4 | 10 |
| 5 | The model has **no memory** — you send the whole conversation every time | 5 | 15 |
| 6 | **Reasoning models** think before they answer — at a price | 6 | 10 |
| 7 | **Structured output**, and where prompting alone is not enough | 7 | 20 |

Run each cell in order. The ✏️ **Practice** cells are yours: change the code, run it, and write one sentence about what you saw.
The whole lab costs a few cents.

## 0 · Setup

In [ ]:
%pip install -q openai transformers tiktoken

In [ ]:
import os, json, math, time
import numpy as np
from openai import OpenAI
from google.colab import userdata

os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
client = OpenAI()

CHAT_MODEL = "gpt-4o-mini"        # small instruction-tuned model: fast and cheap
BIG_MODEL = "gpt-4o"              # larger instruction-tuned model
REASONING_MODEL = "o4-mini"       # reasoning model; if your account lacks it, use any model whose name starts with "o" or "gpt-5"

def chat(messages, model=CHAT_MODEL, show=True, **kwargs):
    """Send a list of messages, print the answer and the token usage, return the full response."""
    if isinstance(messages, str):
        messages = [{"role": "user", "content": messages}]
    t0 = time.time()
    r = client.chat.completions.create(model=model, messages=messages, **kwargs)
    if show:
        print(r.choices[0].message.content)
        u = r.usage
        print(f"\n— {model} · {u.prompt_tokens} prompt + {u.completion_tokens} completion tokens · {time.time() - t0:.1f} s")
    return r

chat("Say hello to Santiago in five words.")

## 1 · A base model only continues text

GPT-2 (OpenAI, 2019) is a **base model**: pre-trained to predict the next token, and nothing else — no instruction tuning, no RLHF.
It is small (124 M parameters), so it runs right here in Colab, and we can look inside: for any prompt it gives a probability for
every one of its 50,257 tokens. This is the softmax at the end of the GPT slides.

In [ ]:
import torch
from transformers import GPT2LMHeadModel, GPT2TokenizerFast

tok = GPT2TokenizerFast.from_pretrained("gpt2")
gpt2 = GPT2LMHeadModel.from_pretrained("gpt2").eval()

def next_tokens(prompt, k=10):
    """The k most likely next tokens after `prompt`, with their probabilities."""
    ids = tok(prompt, return_tensors="pt").input_ids
    with torch.no_grad():
        logits = gpt2(ids).logits[0, -1]
    probs = torch.softmax(logits, dim=-1)
    top = torch.topk(probs, k)
    for p, i in zip(top.values, top.indices):
        print(f"{p.item():6.1%}  {tok.decode(int(i))!r}")

def continue_text(prompt, max_new_tokens=40, temperature=None):
    """Let GPT-2 continue `prompt`. temperature=None means greedy: always take the most likely token."""
    ids = tok(prompt, return_tensors="pt").input_ids
    sampling = dict(do_sample=True, temperature=temperature, top_k=0) if temperature else dict(do_sample=False)
    out = gpt2.generate(ids, max_new_tokens=max_new_tokens, pad_token_id=tok.eos_token_id, **sampling)
    print(repr(tok.decode(out[0][ids.shape[1]:])))

next_tokens("Cheese is a fascinating food because")
print()
continue_text("Cheese is a fascinating food because")

Greedy decoding picks the most likely token every time — and often falls into a loop. Sampling with a temperature
divides the logits by *T* before the softmax: low *T* sharpens the distribution, high *T* flattens it.

In [ ]:
for T in [None, 0.7, 1.5]:
    print(f"temperature = {T or 'greedy'}:")
    continue_text("Cheese is a fascinating food because", temperature=T)
    print()

### ✏️ Practice 1 — what does a base model do with a question?

1. Look at the next-token probabilities for `"The capital of Chile is"`. How confident is GPT-2?
2. Run `continue_text` on the question below a few times, with and without a temperature. Does GPT-2 *answer* it?
3. Write a prompt that makes GPT-2 give a useful answer anyway. *(Hint: base models continue documents. What kind of document
   would contain the answer right after your prompt?)*

In [ ]:
next_tokens("The capital of Chile is")

question = "What is a good cheese for fondue?"
continue_text(question)
continue_text(question, temperature=0.8)

# your prompt:
continue_text("...")

## 2 · An instruction-tuned model answers

The same question to a model that went through supervised fine-tuning and RLHF (the last deck of the morning).
A chat model takes a **list of messages**, each with a role:

- `system` — instructions from the developer: who the assistant is, how it should answer;
- `user` — what the person types;
- `assistant` — what the model said before (we come back to this in part 5).

In [ ]:
chat([
    {"role": "system", "content": "You are a cheese expert assistant."},
    {"role": "user", "content": "What is a good cheese for fondue?"},
])

### ✏️ Practice 2 — the system prompt

Keep the user message the same and change only the system prompt so that the assistant is a terse cheesemonger at a market in
Santiago who answers **in Spanish, in one sentence**. Then try one system prompt that the model does *not* follow. Why not?

In [ ]:
chat([
    {"role": "system", "content": "..."},
    {"role": "user", "content": "What is a good cheese for fondue?"},
])

## 3 · The assistant is still a next-token predictor

Underneath, ChatGPT does exactly what GPT-2 did: it puts a probability on every possible next token and samples one.
The API shows you those probabilities with `logprobs=True`.

In [ ]:
def first_token_options(prompt, model=CHAT_MODEL, k=5):
    """The k most likely first tokens of the model's answer, with their probabilities."""
    r = client.chat.completions.create(model=model, messages=[{"role": "user", "content": prompt}],
                                       logprobs=True, top_logprobs=k, max_tokens=1, temperature=0)
    for alt in r.choices[0].logprobs.content[0].top_logprobs:
        print(f"{math.exp(alt.logprob):6.1%}  {alt.token!r}")

first_token_options("Answer with one word only. Which country does Roquefort come from?")
print()
first_token_options("Answer with one word only. What is the best cheese in the world?")

`temperature` works as it did for GPT-2. With `n=5` the API samples five answers to the same prompt in one call.

In [ ]:
prompt = "Invent a name for a new cheese from the Atacama desert. Reply with the name only."
for T in [0, 1.0, 1.6]:
    r = client.chat.completions.create(model=CHAT_MODEL, messages=[{"role": "user", "content": prompt}], temperature=T, n=5)
    print(f"T={T}:", [c.message.content.strip() for c in r.choices])

### ✏️ Practice 3 — how sure is the model?

1. Find a question where the most likely first token has **less than 50 %** probability. What kind of question is it?
2. For a factual question, would you use temperature 0 or 1.6? And for brainstorming names? Why?

In [ ]:
first_token_options("...")

## 4 · Tokens, context and cost

Models read and write **tokens**, not words (the tokenization slides this morning). Prices and the context window are counted in
tokens too. `tiktoken` is the tokenizer the OpenAI models use.

In [ ]:
import tiktoken
enc = tiktoken.get_encoding("o200k_base")   # the tokenizer of the GPT-4o family

def show_tokens(text):
    ids = enc.encode(text)
    print(f"{len(ids):3d} tokens:", [enc.decode([i]) for i in ids])

show_tokens("Cheese is a fascinating food.")
show_tokens("El queso es un alimento fascinante.")
show_tokens("Tolminc is a hard cheese from the Soča valley.")

### ✏️ Practice 4 — what does a call cost?

Prices change; look up the current ones at [openai.com/api/pricing](https://openai.com/api/pricing) and fill them in below
(USD per million tokens). Then estimate the cost of answering **10,000 customer questions** with each model, if a question is
about 60 tokens with a 200-token system prompt, and an answer is about 150 tokens.

In [ ]:
PRICE = {  # USD per 1M tokens: (input, output) — fill in today's prices
    CHAT_MODEL: (0.15, 0.60),
    BIG_MODEL: (2.50, 10.00),
}
def cost(model, prompt_tokens, completion_tokens):
    p_in, p_out = PRICE[model]
    return (prompt_tokens * p_in + completion_tokens * p_out) / 1e6

for m in PRICE:
    print(m, f"${cost(m, 10_000 * (60 + 200), 10_000 * 150):.2f}")

## 5 · The model has no memory

Every API call is independent. ChatGPT *looks* like it remembers because the app sends the **whole conversation** — every earlier
user and assistant message — with each new question.

In [ ]:
chat("Hi! My name is Valentina and I love goat cheese.")
print("=" * 60)
chat("What is my name, and what cheese should I try tonight?")   # a new call: the model never saw the first message

In [ ]:
history = [
    {"role": "user", "content": "Hi! My name is Valentina and I love goat cheese."},
    {"role": "assistant", "content": "Nice to meet you, Valentina! Goat cheese is wonderful."},
    {"role": "user", "content": "What is my name, and what cheese should I try tonight?"},
]
chat(history)

### ✏️ Practice 5 — build the memory yourself

Complete `Conversation` so that `ask()` sends the history, stores the answer, and keeps only the last `max_turns` exchanges
(the context window is finite, and every token costs). Then have a five-turn conversation with `max_turns=2`. What does it forget?

In [ ]:
class Conversation:
    def __init__(self, system="You are a friendly cheese expert.", max_turns=2):
        self.system = {"role": "system", "content": system}
        self.max_turns = max_turns
        self.history = []            # user and assistant messages, oldest first

    def ask(self, text):
        self.history.append({"role": "user", "content": text})
        # TODO 1: keep only the last self.max_turns exchanges (one exchange = one user + one assistant message)
        # TODO 2: send [self.system] + self.history with chat(..., show=False)
        # TODO 3: append the answer to self.history as an "assistant" message, and return it
        ...

c = Conversation(max_turns=2)
for msg in ["My name is Matías.", "I live in Concepción.", "I am allergic to cow's milk.",
            "Suggest a cheese for me.", "What is my name and where do I live?"]:
    print("👤", msg); print("🤖", c.ask(msg), "\n")

## 6 · Reasoning models think before they answer

A reasoning model is trained (with reinforcement learning) to write a long hidden chain of thought before the answer.
You pay for those **reasoning tokens** even though you do not see them. `reasoning_effort` sets how hard it thinks.

In [ ]:
puzzle = ("A 12 kg wheel of cheese loses 15% of its weight in the first month of ageing, "
          "then 10% of its remaining weight in every month after that. How much does it weigh after 4 months? "
          "Give the final answer in kg with two decimals.")

r1 = chat(puzzle, model=CHAT_MODEL)
print("=" * 60)
r2 = chat(puzzle, model=REASONING_MODEL, reasoning_effort="low")
print("hidden reasoning tokens:", r2.usage.completion_tokens_details.reasoning_tokens)
print("=" * 60)
print("correct answer:", round(12 * 0.85 * 0.9 ** 3, 2), "kg")

### ✏️ Practice 6 — when is thinking worth it?

Find one question where the reasoning model is clearly better, and one where it only adds cost and waiting time.
Try `reasoning_effort="high"` on the first one: how many reasoning tokens does it use now?

In [ ]:
chat("...", model=CHAT_MODEL)
r = chat("...", model=REASONING_MODEL, reasoning_effort="low")
print("reasoning tokens:", r.usage.completion_tokens_details.reasoning_tokens)

## 7 · Structured output — and where prompting alone is not enough

Programs need data, not prose. With `response_format` you give the model a JSON schema, and the answer is guaranteed to match it.

In [ ]:
text = """At the market I tried three cheeses: a crumbly Chanco from the south of Chile, made from cow's milk;
a soft French Crottin de Chavignol made from goat's milk; and a hard, nutty Manchego from Spain, made from sheep's milk."""

schema = {
    "name": "cheeses",
    "strict": True,
    "schema": {
        "type": "object",
        "properties": {"cheeses": {"type": "array", "items": {
            "type": "object",
            "properties": {"name": {"type": "string"}, "country": {"type": "string"},
                           "milk": {"type": "string", "enum": ["cow", "goat", "sheep", "buffalo", "mixed"]},
                           "texture": {"type": "string"}},
            "required": ["name", "country", "milk", "texture"], "additionalProperties": False}}},
        "required": ["cheeses"], "additionalProperties": False},
}
r = chat([{"role": "user", "content": "Extract every cheese from this text:\n" + text}], show=False,
         response_format={"type": "json_schema", "json_schema": schema})
data = json.loads(r.choices[0].message.content)
for c in data["cheeses"]:
    print(c)

Now ask about something the model **cannot** know: a short 2024 book, *Tolminc Cheese* by Pavlos Protopapas, that was never in its
training data. Then give it the text and ask again.

In [ ]:
import urllib.request
q = "According to the book 'Tolminc Cheese' by Pavlos Protopapas (2024), how is Tolminc cheese made? Answer in three sentences."
chat(q)
print("=" * 60)
book = urllib.request.urlopen("https://raw.githubusercontent.com/dlops-io/llm-rag/main/input-datasets/books/Tolminc%20Cheese.txt").read().decode()
print(f"the book is {len(enc.encode(book)):,} tokens long")
chat([{"role": "system", "content": "Answer only from the text provided. If the text does not say, say so."},
      {"role": "user", "content": q + "\n\nText:\n" + book}])

Without the text, the model either admits it does not know — or, worse, invents a confident answer. With the text in the prompt,
it answers from it. That works for one short book. **What if you have eleven books, or a whole company's documents, far more than
fits in a prompt?** That is the 15:30 lecture: retrieval-augmented generation.

### ✏️ Practice 7 — put it together (20 min)

Build a **cheese-pairing assistant** for a Chilean wine shop. It gets a wine and returns JSON:
`{"wine": ..., "cheese": ..., "why": ..., "confidence": "high" | "medium" | "low"}`.

1. Write a system prompt, and add two example exchanges (a user message and the ideal assistant JSON) before the real question —
   this is *few-shot prompting*.
2. Use a JSON schema so the output always parses.
3. Test it on the three wines below. Change one thing (the model, the temperature, or the examples) and compare.
4. One sentence: which of today's ideas — next-token prediction, the system prompt, temperature, no memory, structured output —
   mattered most for getting a good result?

In [ ]:
wines = ["Carmenère from the Colchagua valley", "Sauvignon blanc from Casablanca", "País from the Itata valley"]

# your system prompt, few-shot examples and schema here
messages = [
    {"role": "system", "content": "..."},
]
for w in wines:
    ...

---
### Optional — other providers, same code

Many providers (OpenRouter, Together, Groq, a local Ollama server, …) accept the same API: only `base_url`, the key and the model
names change. With your own key from one of them, this runs open models such as Llama, Mistral or DeepSeek:

```python
other = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=userdata.get("OPENROUTER_API_KEY"))
r = other.chat.completions.create(model="meta-llama/llama-3.3-70b-instruct",
                                  messages=[{"role": "user", "content": "Why does aged cheese taste stronger?"}])
print(r.choices[0].message.content)
```

*Adapted from ELELEM Lab 4 (2025).*